# Học xG từ cú sút World Cup 2022

Nguồn dữ liệu: [StatsBomb Open Data](https://github.com/hudl/open-data). Notebook này là bài học thực hành: đọc dữ liệu → hiểu nhãn → tạo đặc điểm → chia theo trận → so sánh mô hình → kiểm tra xác suất → dự đoán một cú sút.

**xG = 0,20** nghĩa là xác suất thành bàn ước tính 20%. Đây là xác suất cho *một cú sút*, không phải số bàn thắng chắc chắn.

## 1. Mở dữ liệu

Mỗi dòng là một cú sút. `goal` là nhãn 0/1; các cột `distance`, `angle`, `body_part`, `shot_type`, `under_pressure`, `first_time` là đầu vào. `statsbomb_xg` là điểm tham khảo của StatsBomb, **không được dùng để huấn luyện**. Lần đầu hãy chạy lệnh `prepare` trong README để tải 64 trận.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display, Image

here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / 'xg' / 'src' / 'project.py').exists())
sys.path.insert(0, str(root / 'xg' / 'src'))
import project as xg

shots = xg.load_data()
print('Số trận:', shots.match_id.nunique(), '| Số cú sút:', len(shots), '| Số bàn:', int(shots.goal.sum()))
display(shots[['match_id', 'player', 'x', 'y', 'distance', 'angle', 'goal']].head())
print('\nGiá trị thiếu trong các đặc điểm:')
display(shots[xg.FEATURES].isna().sum().to_frame('missing'))

## 2. Nhìn vị trí và tỷ lệ bàn thắng

StatsBomb dùng sân 120 × 80. Đội tấn công về phía khung thành ở x=120; tâm cầu môn y=40. Góc và khoảng cách được tính từ tọa độ *trước cú sút*. Tỷ lệ bàn thắng thấp cho thấy nếu luôn đoán 'không bàn' thì accuracy dễ gây hiểu nhầm; bài này đánh giá **xác suất** bằng Brier score.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(shots.x, shots.y, c=shots.goal, s=10, alpha=.5, cmap='coolwarm')
axes[0].set(xlim=(0, 120), ylim=(0, 80), xlabel='x', ylabel='y', title='Vị trí sút (đỏ = bàn thắng)')
axes[1].hist(shots.distance, bins=25)
axes[1].set(xlabel='Khoảng cách trên hệ tọa độ StatsBomb', ylabel='Số cú sút', title='Phân bố khoảng cách')
plt.tight_layout()
display(fig)
plt.close(fig)
print('Tỷ lệ cú sút thành bàn:', f'{shots.goal.mean():.1%}')
print('Ví dụ x=105, y=40:', xg.shot_geometry(105, 40))

## 3. Huấn luyện và kiểm tra

Mô hình không biết trước cú sút có thành bàn hay không. Chúng ta giữ khoảng 20% **trận** làm test; các cú sút cùng trận không bị chia ra hai tập. Trên train, `GroupKFold` chia 5 lượt để so sánh: tỷ lệ bàn chung (baseline), hồi quy logistic, rừng ngẫu nhiên. Mỗi lượt chỉ học cách điền thiếu, chuẩn hóa, mã hóa từ phần train của lượt đó. Mô hình có **Brier score trung bình thấp nhất** được chọn. Cell này có thể mất vài phút.

In [ ]:
result = xg.train()
print('Mô hình được chọn:', result['model'])
print('Train:', result['train_matches'], 'trận,', result['train_shots'], 'cú sút')
print('Test:', result['test_matches'], 'trận,', result['test_shots'], 'cú sút')
display(pd.read_csv(xg.REPORTS / 'model_comparison.csv'))
display(pd.DataFrame({'mô hình được chọn': result['test'], 'baseline': result['baseline_test'], 'StatsBomb tham khảo': result['statsbomb_reference_test']}))

## 4. Đọc kết quả

**Brier score** = trung bình `(xG - goal)²`; càng thấp càng tốt. **Log loss** phạt mạnh dự đoán quá chắc nhưng sai. **ROC AUC** đo khả năng xếp cú sút thành bàn cao hơn cú sút không thành bàn (0,5 ≈ đoán ngẫu nhiên). Biểu đồ bên trái cho thấy phân bố xG của cú sút có/không bàn. Biểu đồ bên phải so sánh xG trung bình với tỷ lệ bàn thật; các chấm nên gần đường chéo, nhưng khoảng có ít cú sút rất nhiễu. Kết quả test chỉ phản ánh trận khác **trong cùng World Cup 2022**.

In [ ]:
display(Image(filename=str(xg.REPORTS / 'evaluation.png')))
errors = pd.read_csv(xg.REPORTS / 'test_predictions.csv')
display(errors[['player', 'team', 'distance', 'angle', 'goal', 'predicted_xg', 'absolute_error']].head(10))

## 5. Dự đoán một cú sút

Thử sửa tọa độ hoặc kiểu sút rồi chạy lại cell. `warnings` báo khi đầu vào nằm ngoài phạm vi train. Mô hình trả về xác suất, không biết được cú sút tương lai có thành bàn chắc chắn hay không.

In [ ]:
example = {'x': 105.0, 'y': 40.0, 'body_part': 'Right Foot', 'shot_type': 'Open Play', 'under_pressure': False, 'first_time': True}
print(xg.predict(example))